# sPTC on a real GPU, v4: KV pressure and bets that are sometimes needed

In the second T4 run (v3) nothing waited and nothing was preempted, and a
client-side window cap tied the plugin. This run asks what happens **under KV pressure and
a real queue, with bets that are sometimes needed**: does the speculation-aware scheduler
plugin (`sptc_vllm`) beat the client-side cap and vLLM priority? Same setup as v3:
**Qwen3-1.7B (fp16) on one T4**, vLLM pinned to 0.32.0, and the plugin unchanged since run 2
(`BUNDLE_INFO.json` records the commits).

The simulator's predictions were committed before this notebook, commit messages
`prereg: ...`: `results/prereg_v4` (the chosen T4 config), `results/prereg_v4_fallback`
(its fallback, half the KV blocks) and the step-5 addendum `results/prereg_v4b`. The last
cells score the run against them. Simulated answer (`results/pressure`): **the plugin does
not win**, 0.988x the cap's session p50 in `decoys` and 1.000x in `both`.

**The decision rule, on the measured values:** the plugin wins if its session p50
is at least 5% below the cap's at 4 agents, in either workload, with the pressure check
passed. `sweep/summary.md` (section Pressure) computes it.

**Workloads**, 4 agents, the October 19 mix (`map16`, `map_reduce`, `two_stage`,
`side_branch_chain`, 4 turns each, the root stream served by the same vLLM):
- `decoys` (`spec+bad`, as run 2): one never-used decoy per speculative call.
- `both`: spec-ptc's speculative calls are bets too, each with a decoy: both arms of a
  branch launched, one used, so half the bets are needed. A refused bet is run for real
  when the program claims it (spec-ptc's sptc-hints patch, logged as `fallback` with
  `at_claim`); a refused bet the shadow forces stops the shadow for the turn.

| arm | server | speculation | real spec calls | tags | client cap | role |
|---|---|---|---|---|---|---|
| `fcfs-p/nospec` | fcfs_p | off | - | none | - | refuse-all reference for `both` |
| `fcfs-p/spec` | fcfs_p | on, no decoys | early | none | - | no waste; the oracle for `both` |
| `fcfs-p/spec+bad` | fcfs_p | on | early | none | - | denominator |
| `prio-p/spec+bad` | priority_p | on | early | priority_bets | - | |
| `cap-p/spec+bad` | fcfs_p | on | early | none | window:8 | |
| `plugin-p/spec+bad` | plugin_p | on | early | sptc_bets | - | |
| `prio-p/both` | priority_p | on | bet | priority_bets | - | |
| `cap-p/both` | fcfs_p | on | bet | none | window:8 | |
| `plugin-p/both` | plugin_p | on | bet | sptc_bets | - | |

Only bets carry a class (`sptc_bets`, `priority_bets`): the class as the drafted RFC had it.
There is no `fcfs-p/both`: on stock FCFS it is `fcfs-p/spec+bad`. The `_p` servers are
stock FCFS, vLLM priority and the plugin with the T4 config the simulated study chose: `--num-gpu-blocks-override
257 --max-num-seqs 256 --max-model-len 512` (fallback: 129 blocks).

**Settings:** Accelerator **GPU T4**, Internet **on**, Persistence **"Files only"** (so
`/kaggle/working` survives the session's end). **Input:** the dataset made from
`sptc-bundle-v4.zip` (`kaggle/make_bundle.sh`); attach only this bundle. Optional input when
resuming: the `sptc_out_v4.zip` of an earlier session of this notebook, uploaded as a
dataset (finished runs, and the probe's decision, are then kept).

| section | what | estimate |
|---|---|---|
| setup | install vLLM 0.32.0, spec-ptc (sptc-hints), the plugin | ~8 min |
| A | tests: the plugin's local tests, the bench end to end on the fake server (CPU) | ~5 min |
| B | probe: one fcfs_p launch (plus one unmeasured cache-filling launch), 2 runs; picks the config | ~10-15 min |
| C | the pressure sweep: 9 arms, 3 repeats, each server launched twice (ABBA) | ~55 min |
| D | step 5: `both` on the default servers, 6 arms, the same protocol | ~50 min |
| E | report, scoring, zip | ~2 min |
| **total** | | **~2.5 h** |

Estimates: `vllm_bench.py plan` (simulator session times x 1.3, ~20 s start-up per run,
4 min per server launch): upper bounds rather than means. Every run is saved as soon as it
ends and a rerun skips what is finished. `/kaggle/working/sptc_out_v4.zip` (with a copy of
the predictions) is refreshed at the end of each section and every 10 minutes.

In [ ]:
import os, subprocess, sys, time, json, glob, shutil, zipfile

def bundles(top="/kaggle/input"):
    # folders that contain both sptc-sim and spec-ptc, at any depth
    return [root for root, dirs, files in os.walk(top) if "sptc-sim" in dirs and "spec-ptc" in dirs]

def find_bundle(top="/kaggle/input"):
    found = bundles(top)
    if not found:  # zip uploaded but not unpacked: unpack it into /kaggle/working
        for z in glob.glob(f"{top}/**/*.zip", recursive=True):
            if "sptc_out" in os.path.basename(z):
                continue  # an earlier session's output, not a bundle
            with zipfile.ZipFile(z) as f:
                f.extractall("/kaggle/working/_unzipped")
        found = bundles("/kaggle/working/_unzipped")
    v4 = [b for b in found if os.path.isdir(f"{b}/sptc-sim/results/prereg_v4b")]
    return v4[0] if v4 else None

src = find_bundle()
if src is None:
    print("No v4 bundle (sptc-sim/results/prereg_v4b) found. Here is what is under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.count(os.sep) <= 6:
            print(root, dirs[:8], files[:5])
    raise SystemExit("bundle not found: see the listing above")
print("found bundle at", src)
B = "/kaggle/working/sptc-bundle"
shutil.copytree(src, B, dirs_exist_ok=True)  # inputs are read-only

SIM, SPTC = f"{B}/sptc-sim", f"{B}/spec-ptc"
OUT = "/kaggle/working/sptc_out_v4"
os.makedirs(OUT, exist_ok=True)

# resume: an earlier session's output, if attached (finished runs are then skipped)
for z in glob.glob("/kaggle/input/**/sptc_out_v4*.zip", recursive=True):
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/_resume")
for d in glob.glob("/kaggle/input/**/sptc_out_v4", recursive=True) + \
         glob.glob("/kaggle/working/_resume/**/sptc_out_v4", recursive=True):
    print("carrying over", d)
    shutil.copytree(d, OUT, dirs_exist_ok=True)

import threading
_zip_lock = threading.Lock()

def _zip():
    """Copy the predictions and the bundle's commits next to the results; sptc_out_v4 ->
    sptc_out_v4.zip through a temporary file, so the zip on disk is never half-written."""
    with _zip_lock:
        for v in ("prereg_v4", "prereg_v4_fallback", "prereg_v4b"):
            shutil.copytree(f"{SIM}/results/{v}", f"{OUT}/{v}", dirs_exist_ok=True)
        shutil.copy(f"{B}/BUNDLE_INFO.json", OUT)
        tmp = "/kaggle/working/_sptc_out_v4.tmp.zip"
        if os.path.exists(tmp):
            os.remove(tmp)
        r = subprocess.run(["nice", "-n", "19", "zip", "-qr", tmp, "sptc_out_v4"],
                           cwd="/kaggle/working")
        if r.returncode not in (0, 18):  # 18: a file vanished while zipping (a run being saved)
            raise RuntimeError(f"zip exit code {r.returncode}")
        os.replace(tmp, "/kaggle/working/sptc_out_v4.zip")

def save(note=""):
    """Refresh the zip (called at the end of every section)."""
    _zip()
    print(note, "saved:", os.path.getsize("/kaggle/working/sptc_out_v4.zip") // 1024, "KiB")

def _autosave(every_s=600):
    # during the long cells too: a session that ends mid-cell keeps its finished runs
    while True:
        time.sleep(every_s)
        try:
            _zip()
        except Exception as e:  # never kill the thread
            print("autosave failed:", e)

if not any(t.name == "sptc-autosave" for t in threading.enumerate()):
    threading.Thread(target=_autosave, name="sptc-autosave", daemon=True).start()

save("start")
info = json.load(open(f"{B}/BUNDLE_INFO.json"))
print(json.dumps(info, indent=1))
assert info["plugin"]["unchanged_since_run2"], "the plugin changed since run 2: rerun its full tests (v3 section A)"
print(open(f"{SIM}/results/prereg_v4/predictions.md").read()[:700])
!nvidia-smi

In [ ]:
%%bash -s "$SPTC" "$SIM"
pip -q install "vllm==0.32.0" 2>&1 | tail -2
pip -q install pandas numpy pytest 2>&1 | tail -1
pip -q install -e "$1" 2>&1 | tail -1                          # spec-ptc (the sptc-hints version)
pip -q install -e "$2/integrations" --no-deps 2>&1 | tail -1   # sptc_vllm plugin
# Kaggle preinstalls torchaudio built for an older CUDA; vLLM's torch upgrade makes it fail
# at import (transformers imports it), so remove it: nothing here uses audio
pip -q uninstall -y torchaudio 2>&1 | tail -1
python -c "import torchvision.ops, torchvision; print('torchvision', torchvision.__version__, 'OK')"
python -c "import vllm, torch, spec_ptc; print('vllm', vllm.__version__, 'torch', torch.__version__, torch.cuda.get_device_name(0))"
python -c "from spec_ptc.contracts.hints import SpeculationRejected; print('spec-ptc with the sptc-hints patch: OK')"

## A. Tests (CPU; no server running, ~5 min)

1. The plugin's local tests (`integrations/tests`). Where vLLM is installed, as here, the
   tests on a stand-in of vLLM 0.32's scheduler skip, and the ones that need the vendored
   SGLang sources skip (they are not in the bundle).
2. The bench against the fake OpenAI server (`kaggle/tests`), end to end with spec-ptc's
   Harness: among them the pressure preset (each arm's tags, the speculative calls sent as
   bets in `both`, a refused bet re-run once at its claim, one KV config per output
   directory) and the probe.

Not run, unlike v3: the scheduler overhead bench and the vLLM-dependent `tests_vllm` (the
157-case differential against stock vLLM). The plugin has not changed since run 2, which
passed them: the setup cell checked `BUNDLE_INFO.json` (`integrations/sptc_vllm` unchanged
since run 2's bundle, `53722e0`).

In [ ]:
!cd {SIM}/integrations && python -m pytest -q -p no:cacheprovider tests 2>&1 | tee {OUT}/tests_plugin.txt | tail -5
!cd {SIM} && SPTC_SPEC_PTC={SPTC} python -m pytest -q -p no:cacheprovider kaggle/tests 2>&1 | tee {OUT}/tests_bench.txt | tail -8
save("A tests")

## B. Probe (~10-15 min)

One launch of the `fcfs_p` server on that config, then `fcfs-p/spec` and
`fcfs-p/spec+bad` at 4 agents, once each (after the warm-up pass), into `probe/`. The rule,
fixed in `results/prereg_v4/predictions.md`: **if `fcfs-p/spec+bad` shows neither a
preemption (`vllm:num_preemptions`) nor a request waiting in any 1-second sample, the sweep
runs on the pre-registered fallback** (`--num-gpu-blocks-override 129`, half the blocks) and
is scored against `results/prereg_v4_fallback`; otherwise on the chosen config, scored
against `results/prereg_v4`.

The cell prints both arms' measured preemptions, max waiting, queue (share of 1-second
samples with a request waiting) and KV peak next to the predicted ones, and writes the
decision to `pressure_config.json`; a rerun, or a resumed session, reuses it. The session's
first launch compiles fresh: the bench stops it unmeasured and launches again
(`probe/sweep/launches.jsonl`, role `cache_fill`). Every sweep on the `_p` servers takes
`--max-model-len 512` in `--serve-cmd` (the bench refuses to start without it), and an
output directory holds one KV config only (`sweep/pressure.json`).

In [ ]:
BENCH = f"python {SIM}/kaggle/vllm_bench.py"
COMMON = f"--model Qwen/Qwen3-1.7B --spec-ptc {SPTC} --launch"
sys.path.insert(0, f"{SIM}/kaggle")
import vllm_bench as vb
SERVE_P = vb.PRESSURE_SERVE_CMD  # vllm serve ... --max-model-len 512 ...
CHOICE = f"{OUT}/pressure_config.json"
!nvidia-smi --query-gpu=memory.used --format=csv   # must be ~0: no other server running
if os.path.exists(CHOICE):
    verdict = json.load(open(CHOICE))
    print("the probe already decided:", verdict["config"])
else:
    !{BENCH} sweep --preset probe {COMMON} --out {OUT}/probe --serve-cmd "{SERVE_P}" 2>&1 | grep -v "^INFO" | tail -30
    verdict = vb.probe_verdict(f"{OUT}/probe")
    if verdict["config"] is None:
        raise SystemExit(f"probe incomplete: {verdict.get('why')}; rerun this cell")
    json.dump(verdict, open(CHOICE, "w"), indent=1)
PRESSURE = verdict["config"]
PREREG = "prereg_v4" if PRESSURE == "chosen" else "prereg_v4_fallback"
pred = {x["arm"]: x for x in json.load(open(f"{SIM}/results/prereg_v4/predictions.json"))["sets"]["P1"]["pressure"]}
for arm, m in verdict["arms"].items():
    p = pred[arm]
    print(f"{arm:16s} measured  preemptions {m['preemptions']}, max waiting {m['max_waiting']}, "
          f"queue {m['waiting_frac']}, KV peak {m['max_kv']}")
    print(f"{'':16s} predicted preemptions {p['preemptions']:.0f}, max waiting {p['max_waiting']:.0f}, "
          f"queue {p['queue_frac']:.2f} (of the time), KV peak {p['max_kv']:.2f}")
print(f"pressure: {verdict['pressure']} -> the {PRESSURE} config: {' '.join(vb.pressure_flags(PRESSURE))} "
      f"--max-model-len {vb.PRESSURE_MAX_LEN}; scored against results/{PREREG}")
save("B probe")

## C. The pressure sweep (~55 min)

`vllm_bench.py sweep --preset pressure --agents 4` on the config the probe chose: the nine
arms above, 3 repeats, servers `fcfs_p`, `priority_p`, `plugin_p` and then in reverse
(ABBA: repeat r runs on launch r % 2). As in v3: no measurement on a fresh compile (all
three servers share one compiled artifact), 3 warm-up calls and one throwaway 1-agent run
after every launch (`sweep/warm/`), the same prompts on every arm (arms that share a launch
are kept apart in the prefix cache by vLLM's `cache_salt`), and every launch's compile and
GPU state in `sweep/launches.jsonl`.

The **Pressure** section of `sweep/summary.md` gives, per arm, session p50/p99 and root
TPOT p50/p99 as ratios to `fcfs-p/spec+bad`; the shares of decoys and of real bets refused
and dropped; **bets per call** (spec-ptc bets again on a call whose bet failed: a re-peek,
then its statement; the simulator sends one bet per call, see `results/prereg_v4b`, "Found
after prereg_v4"); calls re-run at their claim; sessions whose shadow stopped at a refused
bet; preemptions, max waiting, queue and KV peak; and the decision rule with its pressure
check (thresholds: on `fcfs-p/spec+bad` >= 16 preemptions per run or a queue in
>= 25% of samples, and on `fcfs-p/spec` <= 2 and <= 5%).

In [ ]:
!{BENCH} plan --preset pressure --predictions {SIM}/results/{PREREG}/predictions.json --out {OUT}/plan 2>&1 | head -1
!{BENCH} sweep --preset pressure --agents 4 --pressure-config {PRESSURE} {COMMON} --out {OUT} --serve-cmd "{SERVE_P}" 2>&1 | grep -v "^INFO" | tail -60
!{BENCH} report --out {OUT} > /dev/null 2>&1
!sed -n '/^## Pressure/,/^## Baselines/p' {OUT}/sweep/summary.md
save("C pressure")

## D. Step 5: `both` on the default servers (~50 min)

Step 5, run because the simulated study predicted a 47% effect (over the 5% set
for it): the `both` workload on the default servers (default KV, `--max-model-len 8192`),
where nothing waits. It shows what a fixed window of 8 costs when half the bets are needed.
Arms, 4 agents, 3 repeats, two launches per server: `fcfs-d/nospec` (no speculation),
`fcfs-d/spec` (only the needed bets: the oracle), `fcfs-d/spec+bad` (every bet admitted:
on stock FCFS, `both` and `decoys` are the same), `prio-d/both`, `cap-d/both`,
`plugin-d/both`. Pre-registered in `results/prereg_v4b` (written after v4, before this run).
Not part of the decision rule: it has no pressure. Set `RUN_STEP5 = False` to skip.

In [ ]:
RUN_STEP5 = True
if RUN_STEP5:
    !{BENCH} plan --preset both_default --predictions {SIM}/results/prereg_v4b/predictions.json --out {OUT}/plan 2>&1 | head -1
    !{BENCH} sweep --preset both_default --agents 4 {COMMON} --out {OUT} 2>&1 | grep -v "^INFO" | tail -40
    !{BENCH} report --out {OUT} > /dev/null 2>&1
    !sed -n '/^## Pressure/,/^## Baselines/p' {OUT}/sweep/summary.md | grep -E "^\| (arm|---|[a-z]+-d/)|default servers"
    save("D step 5")

## E. Score against the pre-registered predictions, and download

`experiments/prereg_compare.py` applies the rules fixed in the predictions (v3's,
unchanged) to whatever ran: `results/prereg_v4` (or `prereg_v4_fallback`, after a fallback)
for the pressure arms, and `results/prereg_v4b` for step 5. Every claim is supported,
refuted, unresolved, invalid (host CPU saturated or a failed run) or not run. It also
checks the prediction files' sha256 against their prereg commits, and adds the strict
(launch-level CI) verdict, per-pattern ratios and a work check per ratio claim. Output:
`sptc_out_v4/prereg_compare.md` and `.json`, next to `sweep/summary.md`.

The decision rule is applied to the measured values (`sweep/summary.md`, Pressure), not
to the claims' verdicts. Read the `both` claims next to the Pressure table's bets per
call. Then download `/kaggle/working/sptc_out_v4.zip`.

In [ ]:
!{BENCH} report --out {OUT} 2>&1 | tail -3
!python {SIM}/experiments/prereg_compare.py --out {OUT} --prereg {SIM}/results/{PREREG} --also {SIM}/results/prereg_v4b 2>&1 | head -90

In [ ]:
save("final")
!ls -la /kaggle/working/sptc_out_v4.zip